In [31]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import duckdb
import pandas as pd
from IPython.display import display, HTML

# 1. Locate and load the environment settings
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
INGESTION_DIR = next((c for c in _candidates if (c / ".env").exists()), None)
assert INGESTION_DIR is not None, (
    f"couldn't find services/ingestion/.env starting from cwd={Path.cwd()}"
)
load_dotenv(INGESTION_DIR / ".env")
sys.path.insert(0, str(INGESTION_DIR))

from app.core.config import get_settings  # noqa: E402

# 2. Resolve database path
settings = get_settings()
db_path = Path(settings.duckdb_staging_dir)
if not db_path.is_absolute():
    db_path = INGESTION_DIR / db_path
db_path = db_path / "landed.duckdb"

assert db_path.exists(), f"No staging file found at {db_path} -- run an ingest first."

# %% [markdown]
# ## 1. Summary of Tables and Row Counts

# %%
with duckdb.connect(str(db_path), read_only=True) as conn:
    tables = conn.execute("SELECT table_name FROM duckdb_tables();").fetchall()

    data = []
    for (table_name,) in tables:
        count = conn.execute(f'SELECT count(*) FROM "{table_name}"').fetchone()[0]
        data.append({"table_name": table_name, "row_count": count})

    df_summary = (
        pd.DataFrame(data)
        .sort_values(by="row_count", ascending=False)
        .reset_index(drop=True)
    )

display(df_summary)

# %% [markdown]
# ## 2. Detailed EDA: Schema & Preview
# Run the cell below to automatically fetch column names, data types, and preview the first 5 rows for every table in the database.

# %%
with duckdb.connect(str(db_path), read_only=True) as conn:
    for _, row in df_summary.iterrows():
        t_name = row["table_name"]
        t_rows = row["row_count"]

        display(HTML(f"<h3>📦 Table: <code>{t_name}</code> ({t_rows:,} rows)</h3>"))

        # # Get schema info
        # schema_df = conn.execute(f'DESCRIBE "{t_name}"').fetchdf()
        # display(schema_df)

        # Preview top 5 rows
        display(HTML("<strong>Preview (Top 5 rows):</strong>"))
        preview_df = conn.execute(f'SELECT * FROM "{t_name}" LIMIT 5').fetchdf()
        display(preview_df)

        print("-" * 80)

,table_name,row_count
0,aemo_nem_dispatch,531285
1,openelectricity_mix,213720
2,aemo_wem_dispatch,106273
3,bom_observations,53304
4,aemo_holidays,180


,ts,region,demand_mw,price_mwh,source,ingested_at,ingest_run_id,_ingest_run_id,coal_mw,gas_mw,hydro_mw,wind_mw,solar_utility_mw,solar_rooftop_mw,battery_mw,net_import_mw
0,2026-06-28 20:05:00+06:00,TAS1,1031.37,70.22000,aemo_nem,2026-08-06 14:26:45.559676+06:00,c8209f1a-ddb6-47f4-b3d2-319f204e2388,ed6b81b9-f547-4a49-b369-4e96e0755a7b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-06-28 20:05:00+06:00,QLD1,5879.30,67.75000,aemo_nem,2026-08-06 14:26:45.559676+06:00,c8209f1a-ddb6-47f4-b3d2-319f204e2388,ed6b81b9-f547-4a49-b369-4e96e0755a7b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-06-28 20:05:00+06:00,VIC1,5409.86,11.67000,aemo_nem,2026-08-06 14:26:45.559676+06:00,c8209f1a-ddb6-47f4-b3d2-319f204e2388,ed6b81b9-f547-4a49-b369-4e96e0755a7b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-06-28 20:05:00+06:00,NSW1,7977.02,79.05000,aemo_nem,2026-08-06 14:26:45.559676+06:00,c8209f1a-ddb6-47f4-b3d2-319f204e2388,ed6b81b9-f547-4a49-b369-4e96e0755a7b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-06-28 20:05:00+06:00,SA1,1539.49,41.96523,aemo_nem,2026-08-06 14:26:45.559676+06:00,c8209f1a-ddb6-47f4-b3d2-319f204e2388,ed6b81b9-f547-4a49-b369-4e96e0755a7b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


--------------------------------------------------------------------------------


,ts,battery_discharge_mw,battery_charge_mw,biomass_mw,coal_mw,distillate_mw,gas_mw,hydro_mw,pumped_hydro_mw,solar_rooftop_mw,...,region,total_generation_mw,total_renewable_mw,demand_mw,price_mwh,intensity_kg_per_mwh,source,ingested_at,ingest_run_id,_ingest_run_id
0,2026-07-01 16:00:00+06:00,132.6072,363.8832,23.2122,5061.1551,0.0,43.67,172.9213,0.0,1827.985,...,NSW1,10183.5687,4582.2532,<NA>,<NA>,0.448782,openelectricity,2026-08-05 22:01:46.863771+06:00,c1acc1ba-0cb5-447c-b28d-28c8b6127815,819da310-e741-4a87-9b43-0877bb651609
1,2026-07-01 16:05:00+06:00,260.6815,240.4171,23.1422,5020.5538,0.0,43.71,204.3357,0.0,1827.985,...,NSW1,10191.8805,4626.5181,<NA>,<NA>,0.445032,openelectricity,2026-08-05 22:01:46.863771+06:00,c1acc1ba-0cb5-447c-b28d-28c8b6127815,819da310-e741-4a87-9b43-0877bb651609
2,2026-07-01 16:10:00+06:00,-397.6665,399.3063,23.3822,4941.1575,0.0,43.97,170.0763,0.0,1827.985,...,NSW1,9569.7096,4582.9423,<NA>,<NA>,0.466098,openelectricity,2026-08-05 22:01:46.863771+06:00,c1acc1ba-0cb5-447c-b28d-28c8b6127815,819da310-e741-4a87-9b43-0877bb651609
3,2026-07-01 16:15:00+06:00,-388.2119,399.6667,23.0922,4809.3676,0.0,43.38,149.5140,0.0,1827.985,...,NSW1,9431.6107,4567.4083,<NA>,<NA>,0.460096,openelectricity,2026-08-05 22:01:46.863771+06:00,c1acc1ba-0cb5-447c-b28d-28c8b6127815,819da310-e741-4a87-9b43-0877bb651609
4,2026-07-01 16:20:00+06:00,-416.6562,416.6806,23.1922,4738.1018,0.0,43.53,199.7734,0.0,1827.985,...,NSW1,9330.7845,4549.1283,<NA>,<NA>,0.458491,openelectricity,2026-08-05 22:01:46.863771+06:00,c1acc1ba-0cb5-447c-b28d-28c8b6127815,819da310-e741-4a87-9b43-0877bb651609


--------------------------------------------------------------------------------


...

--------------------------------------------------------------------------------


,ts,station_id,region,temp_c,apparent_temp_c,dew_point_c,humidity_pct,wind_speed_kmh,wind_direction_deg,wind_gust_kmh,pressure_hpa,rain_since_9am_mm,cloud_oktas,source,ingested_at,ingest_run_id,_ingest_run_id
0,2026-01-08 06:00:00+06:00,066037,NSW1,30.9,34.3,18.1,46,9.0,53,25.6,1015.1,<NA>,0.0,bom,2026-08-05 20:24:24.105778+06:00,ecb7f5b0-b096-4b76-a894-ab7f3604371b,17951118-d151-4fc7-b10d-90a132a89873
1,2026-01-08 07:00:00+06:00,066037,NSW1,32.5,36.6,18.9,44,11.2,58,32.4,1014.5,<NA>,0.0,bom,2026-08-05 20:24:24.105778+06:00,ecb7f5b0-b096-4b76-a894-ab7f3604371b,17951118-d151-4fc7-b10d-90a132a89873
2,2026-01-08 08:00:00+06:00,066037,NSW1,33.3,37.6,19.5,44,13.5,65,37.1,1013.9,<NA>,0.0,bom,2026-08-05 20:24:24.105778+06:00,ecb7f5b0-b096-4b76-a894-ab7f3604371b,17951118-d151-4fc7-b10d-90a132a89873
3,2026-01-08 09:00:00+06:00,066037,NSW1,33.1,36.9,19.4,44,16.1,62,42.5,1013.2,<NA>,0.1,bom,2026-08-05 20:24:24.105778+06:00,ecb7f5b0-b096-4b76-a894-ab7f3604371b,17951118-d151-4fc7-b10d-90a132a89873
4,2026-01-08 10:00:00+06:00,066037,NSW1,32.8,35.1,17.7,41,18.0,53,46.4,1012.8,<NA>,0.5,bom,2026-08-05 20:24:24.105778+06:00,ecb7f5b0-b096-4b76-a894-ab7f3604371b,17951118-d151-4fc7-b10d-90a132a89873


--------------------------------------------------------------------------------


,date,region,holiday_name,is_workday,source,ingested_at,ingest_run_id,_ingest_run_id
0,2030-01-01,NSW1,New Year's Day,False,aemo_holidays,2026-08-05 20:26:48.005300+06:00,d8ee67ca-28eb-4866-9dfa-a30f7cdead26,3b3b8a9a-bc0f-42cf-94cf-b408bf92fc94
1,2030-01-26,NSW1,Australia Day,False,aemo_holidays,2026-08-05 20:26:48.005352+06:00,ee896ae2-68fb-4ae7-b660-d21079f803c8,3b3b8a9a-bc0f-42cf-94cf-b408bf92fc94
2,2030-04-25,NSW1,Anzac Day,False,aemo_holidays,2026-08-05 20:26:48.005369+06:00,347cd951-16c1-4e5d-8b50-e41aa2d39fe0,3b3b8a9a-bc0f-42cf-94cf-b408bf92fc94
3,2030-12-25,NSW1,Christmas Day,False,aemo_holidays,2026-08-05 20:26:48.005383+06:00,03f90eb4-5307-40f6-b2c1-25613917e482,3b3b8a9a-bc0f-42cf-94cf-b408bf92fc94
4,2030-12-26,NSW1,Boxing Day,False,aemo_holidays,2026-08-05 20:26:48.005398+06:00,90bb3fd3-670e-4cf2-8ba3-93162946f443,3b3b8a9a-bc0f-42cf-94cf-b408bf92fc94


--------------------------------------------------------------------------------


# Data Frequency

## BOM — Daily Frequency

- **Granularity:** 5 minutes
- **Readings per station per day:** 288
- **Number of stations:** 6
- **Total readings per day:** 1,728

### Calculation

- 24 hours × 60 minutes ÷ 5 minutes = **288 readings per station per day**
- 288 readings × 6 stations = **1,728 readings per day**

In [34]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import duckdb
import pandas as pd
from IPython.display import display, HTML

# 1. Locate and load the environment settings
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
INGESTION_DIR = next((c for c in _candidates if (c / ".env").exists()), None)
assert INGESTION_DIR is not None, (
    f"couldn't find services/ingestion/.env starting from cwd={Path.cwd()}"
)
load_dotenv(INGESTION_DIR / ".env")
sys.path.insert(0, str(INGESTION_DIR))

from app.core.config import get_settings  # noqa: E402

# 2. Resolve database path
settings = get_settings()
db_path = Path(settings.duckdb_staging_dir)
if not db_path.is_absolute():
    db_path = INGESTION_DIR / db_path
db_path = db_path / "landed.duckdb"

assert db_path.exists(), f"No staging file found at {db_path} -- run an ingest first."

# --- CONFIGURATION ---
TABLE_NAME = "bom_observations"
START_DATE = "2026-07-01"
END_DATE = "2026-08-06"
EXPECTED_DAILY_COUNT = 1728
# ---------------------

# %% [markdown]
# ## Completeness Check Query

# %%
query = f"""
SELECT
    cal.record_date,
    COUNT(d.ts) AS total_count
FROM (
    SELECT unnest(generate_series(
        '{START_DATE}'::timestamp,
        '{END_DATE}'::timestamp,
        '1 day'::interval
    ))::date AS record_date
) cal
LEFT JOIN {TABLE_NAME} d
       ON DATE(d.ts) = cal.record_date
      AND d.ts >= '{START_DATE} 00:00:00+00'
      AND d.ts < '{END_DATE} 00:00:00+00'
GROUP BY cal.record_date
HAVING COUNT(d.ts) < {EXPECTED_DAILY_COUNT}
ORDER BY cal.record_date ASC;
"""

with duckdb.connect(str(db_path), read_only=True) as conn:
    df_gaps = conn.execute(query).fetchdf()

display(
    HTML(
        f"<h3>⚠️ Incomplete Days Found (< {EXPECTED_DAILY_COUNT} records): {len(df_gaps)}</h3>"
    )
)
display(df_gaps)

,record_date,total_count
0,2026-07-01,108
1,2026-07-02,144
2,2026-07-03,144
3,2026-07-04,144
4,2026-07-05,144
5,2026-07-06,144
6,2026-07-07,144
7,2026-07-08,144
8,2026-07-09,144
9,2026-07-10,144


# Data Frequency

## OpenElectricity / OpenNEM (`ds-openelectricity`)

- **Regions:** 5 NEM regions
- **Primary granularity:** 30 minutes

| Granularity | Readings per Region / Day | × 5 Regions / Day | × 1 Year |
|---|---:|---:|---:|
| 5 minutes | 288 | 1,440 | 525,600 |
| 15 minutes | 96 | 480 | 175,200 |
| 30 minutes | 48 | 240 | 87,600 |
| Hourly | 24 | 120 | 43,800 |
| Daily | 1 | 5 | 1,825 |

### Calculation

- **5 minutes:** 288 × 5 × 365 = **525,600 readings/year**
- **15 minutes:** 96 × 5 × 365 = **175,200 readings/year**
- **30 minutes:** 48 × 5 × 365 = **87,600 readings/year**
- **Hourly:** 24 × 5 × 365 = **43,800 readings/year**
- **Daily:** 1 × 5 × 365 = **1,825 readings/year**

In [36]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import duckdb
import pandas as pd
from IPython.display import display, HTML

# 1. Locate and load the environment settings
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
INGESTION_DIR = next((c for c in _candidates if (c / ".env").exists()), None)
assert INGESTION_DIR is not None, (
    f"couldn't find services/ingestion/.env starting from cwd={Path.cwd()}"
)
load_dotenv(INGESTION_DIR / ".env")
sys.path.insert(0, str(INGESTION_DIR))

from app.core.config import get_settings  # noqa: E402

# 2. Resolve database path
settings = get_settings()
db_path = Path(settings.duckdb_staging_dir)
if not db_path.is_absolute():
    db_path = INGESTION_DIR / db_path
db_path = db_path / "landed.duckdb"

assert db_path.exists(), f"No staging file found at {db_path} -- run an ingest first."

# --- CONFIGURATION ---
TABLE_NAME = "aemo_nem_dispatch"
START_DATE = "2026-07-01"
END_DATE = "2026-08-06"
EXPECTED_DAILY_COUNT = 1440
# ---------------------

# %% [markdown]
# ## Completeness Check Query

# %%
query = f"""
SELECT
    cal.record_date,
    COUNT(d.ts) AS total_count
FROM (
    SELECT unnest(generate_series(
        '{START_DATE}'::timestamp,
        '{END_DATE}'::timestamp,
        '1 day'::interval
    ))::date AS record_date
) cal
LEFT JOIN {TABLE_NAME} d
       ON DATE(d.ts) = cal.record_date
      AND d.ts >= '{START_DATE} 00:00:00+00'
      AND d.ts < '{END_DATE} 00:00:00+00'
GROUP BY cal.record_date
HAVING COUNT(d.ts) < {EXPECTED_DAILY_COUNT}
ORDER BY cal.record_date ASC;
"""

with duckdb.connect(str(db_path), read_only=True) as conn:
    df_gaps = conn.execute(query).fetchdf()

display(
    HTML(
        f"<h3>⚠️ Incomplete Days Found (< {EXPECTED_DAILY_COUNT} records): {len(df_gaps)}</h3>"
    )
)
display(df_gaps)

,record_date,total_count
0,2026-07-01,1080
1,2026-08-04,1205
2,2026-08-05,0
3,2026-08-06,30


# Data Frequency

## AEMO NEM (`ds-aemo-nem`)

- **Regions:** 5 NEM regions
- **Primary granularity:** 5 minutes

| Granularity | Readings per Region / Day | × 5 Regions / Day | × 1 Year |
|---|---:|---:|---:|
| 5 minutes | 288 | 1,440 | 525,600 |
| 15 minutes | 96 | 480 | 175,200 |
| 30 minutes | 48 | 240 | 87,600 |
| Hourly | 24 | 120 | 43,800 |
| Daily | 1 | 5 | 1,825 |

### Calculation

- **5 minutes:** 288 × 5 × 365 = **525,600 readings/year**
- **15 minutes:** 96 × 5 × 365 = **175,200 readings/year**
- **30 minutes:** 48 × 5 × 365 = **87,600 readings/year**
- **Hourly:** 24 × 5 × 365 = **43,800 readings/year**
- **Daily:** 1 × 5 × 365 = **1,825 readings/year**

In [37]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import duckdb
import pandas as pd
from IPython.display import display, HTML

# 1. Locate and load the environment settings
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
INGESTION_DIR = next((c for c in _candidates if (c / ".env").exists()), None)
assert INGESTION_DIR is not None, (
    f"couldn't find services/ingestion/.env starting from cwd={Path.cwd()}"
)
load_dotenv(INGESTION_DIR / ".env")
sys.path.insert(0, str(INGESTION_DIR))

from app.core.config import get_settings  # noqa: E402

# 2. Resolve database path
settings = get_settings()
db_path = Path(settings.duckdb_staging_dir)
if not db_path.is_absolute():
    db_path = INGESTION_DIR / db_path
db_path = db_path / "landed.duckdb"

assert db_path.exists(), f"No staging file found at {db_path} -- run an ingest first."

# --- CONFIGURATION ---
TABLE_NAME = "aemo_nem_dispatch"
START_DATE = "2025-08-01"
END_DATE = "2026-08-04"
EXPECTED_DAILY_COUNT = 1440
# ---------------------

# %% [markdown]
# ## Completeness Check Query

# %%
query = f"""
SELECT
    cal.record_date,
    COUNT(d.ts) AS total_count
FROM (
    SELECT unnest(generate_series(
        '{START_DATE}'::timestamp,
        '{END_DATE}'::timestamp,
        '1 day'::interval
    ))::date AS record_date
) cal
LEFT JOIN {TABLE_NAME} d
       ON DATE(d.ts) = cal.record_date
      AND d.ts >= '{START_DATE} 00:00:00+00'
      AND d.ts < '{END_DATE} 00:00:00+00'
GROUP BY cal.record_date
HAVING COUNT(d.ts) < {EXPECTED_DAILY_COUNT}
ORDER BY cal.record_date ASC;
"""

with duckdb.connect(str(db_path), read_only=True) as conn:
    df_gaps = conn.execute(query).fetchdf()

display(
    HTML(
        f"<h3>⚠️ Incomplete Days Found (< {EXPECTED_DAILY_COUNT} records): {len(df_gaps)}</h3>"
    )
)
display(df_gaps)

,record_date,total_count
0,2025-08-01,1080
1,2026-03-10,1335
2,2026-08-04,360


# Data Frequency

## AEMO WEM (`ds-aemo-wem`)

- **Region:** 1 WEM region
- **Primary granularity:** 30 minutes

| Granularity | Readings per Region / Day | × 1 Region / Day | × 1 Year |
|---|---:|---:|---:|
| 5 minutes | 288 | 288 | 105,120 |
| 15 minutes | 96 | 96 | 35,040 |
| 30 minutes | 48 | 48 | 17,520 |
| Hourly | 24 | 24 | 8,760 |
| Daily | 1 | 1 | 365 |

### Calculation

- **5 minutes:** 288 × 1 × 365 = **105,120 readings/year**
- **15 minutes:** 96 × 1 × 365 = **35,040 readings/year**
- **30 minutes:** 48 × 1 × 365 = **17,520 readings/year**
- **Hourly:** 24 × 1 × 365 = **8,760 readings/year**
- **Daily:** 1 × 1 × 365 = **365 readings/year**

In [38]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import duckdb
import pandas as pd
from IPython.display import display, HTML

# 1. Locate and load the environment settings
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
INGESTION_DIR = next((c for c in _candidates if (c / ".env").exists()), None)
assert INGESTION_DIR is not None, (
    f"couldn't find services/ingestion/.env starting from cwd={Path.cwd()}"
)
load_dotenv(INGESTION_DIR / ".env")
sys.path.insert(0, str(INGESTION_DIR))

from app.core.config import get_settings  # noqa: E402

# 2. Resolve database path
settings = get_settings()
db_path = Path(settings.duckdb_staging_dir)
if not db_path.is_absolute():
    db_path = INGESTION_DIR / db_path
db_path = db_path / "landed.duckdb"

assert db_path.exists(), f"No staging file found at {db_path} -- run an ingest first."

# --- CONFIGURATION ---
TABLE_NAME = "aemo_wem_dispatch"
START_DATE = "2025-08-01"
END_DATE = "2026-08-04"
EXPECTED_DAILY_COUNT = 288
# ---------------------

# %% [markdown]
# ## Completeness Check Query

# %%
query = f"""
SELECT
    cal.record_date,
    COUNT(d.ts) AS total_count
FROM (
    SELECT unnest(generate_series(
        '{START_DATE}'::timestamp,
        '{END_DATE}'::timestamp,
        '1 day'::interval
    ))::date AS record_date
) cal
LEFT JOIN {TABLE_NAME} d
       ON DATE(d.ts) = cal.record_date
      AND d.ts >= '{START_DATE} 00:00:00+00'
      AND d.ts < '{END_DATE} 00:00:00+00'
GROUP BY cal.record_date
HAVING COUNT(d.ts) < {EXPECTED_DAILY_COUNT}
ORDER BY cal.record_date ASC;
"""

with duckdb.connect(str(db_path), read_only=True) as conn:
    df_gaps = conn.execute(query).fetchdf()

display(
    HTML(
        f"<h3>⚠️ Incomplete Days Found (< {EXPECTED_DAILY_COUNT} records): {len(df_gaps)}</h3>"
    )
)
display(df_gaps)

,record_date,total_count
0,2025-08-01,216
1,2026-08-04,72
